<a href="https://www.kaggle.com/code/ipshitajafrin/covid19-detection?scriptVersionId=292135507" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

### Load Dataset

In [ ]:
import os

# Define the base path to the dataset (in the same folder as the notebook)
base_path = '/kaggle/input/covid-19-radiography/COVID-19_Radiography_Dataset'

# List of expected class labels from the prompt
expected_classes = ['COVID', 'Normal', 'Viral Pneumonia', 'Lung_Opacity']

# Dictionary to store class counts
class_counts = {}

print(f"Inspecting dataset directory: {base_path}\n")

# Iterate through each expected class subdirectory
for class_name in expected_classes:
    # Images are in a nested 'images' subdirectory
    class_path = os.path.join(base_path, class_name, 'images')
    if os.path.exists(class_path) and os.path.isdir(class_path):
        # Count image files (assuming common image extensions like .png, .jpg, .jpeg)
        image_count = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        class_counts[class_name] = image_count
    else:
        class_counts[class_name] = 0 # Or handle as an error if directory not found
        print(f"Warning: Directory for class '{class_name}' not found at {class_path}")

# Print the results
print("Dataset Structure and Image Counts:")
for class_name, count in class_counts.items():
    print(f"- {class_name}: {count} images")

In [ ]:
# Initialize an empty dictionary to store refined class counts
refined_class_counts = {}

print(f"Refining inspection of dataset directory: {base_path}\n")

# Iterate through each expected class subdirectory
for class_name in expected_classes:
    current_class_root_path = os.path.join(base_path, class_name)

    if not (os.path.exists(current_class_root_path) and os.path.isdir(current_class_root_path)):
        print(f"Warning: Class root directory '{class_name}' not found at {current_class_root_path}. Skipping.")
        refined_class_counts[class_name] = 0
        continue

    current_class_total_images = 0
    print(f"Processing class: {class_name} at {current_class_root_path}")

    # List all entries within the current class root path
    for entry in os.listdir(current_class_root_path):
        potential_image_dir = os.path.join(current_class_root_path, entry)

        # Check if the entry is a directory (e.g., 'images' or 'TRAIN', 'TEST')
        if os.path.isdir(potential_image_dir):
            # Count image files within this subdirectory
            image_files_in_subdir = [f for f in os.listdir(potential_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            current_class_total_images += len(image_files_in_subdir)
            print(f"  Found {len(image_files_in_subdir)} images in {entry}/")

    # Store the total count for the current class
    refined_class_counts[class_name] = current_class_total_images

# Print the refined results
print("\nRefined Dataset Structure and Image Counts:")
for class_name, count in refined_class_counts.items():
    print(f"- {class_name}: {count} images")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract class names and counts from the refined_class_counts dictionary
class_names = list(refined_class_counts.keys())
image_counts = list(refined_class_counts.values())

# Create the bar plot
plt.figure(figsize=(10, 6))
sns.barplot(x=class_names, y=image_counts, palette='viridis')

# Add title and labels
plt.title('Dataset Class Distribution')
plt.xlabel('Class')
plt.ylabel('Number of Images')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Display the plot
plt.tight_layout()
plt.show()


### Display Sample Images

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

print("Necessary libraries imported: os, random, matplotlib.pyplot, PIL.Image")

In [ ]:
def display_sample_images(class_name, num_samples=4):
    class_images_path = os.path.join(base_path, class_name, 'images')

    if not os.path.exists(class_images_path):
        print(f"Warning: Image directory for class '{class_name}' not found at {class_images_path}")
        return

    # List all image files in the directory
    image_files = [f for f in os.listdir(class_images_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if not image_files:
        print(f"No image files found in '{class_name}' at {class_images_path}")
        return

    # Randomly select sample images
    selected_images = random.sample(image_files, min(len(image_files), num_samples))

    plt.figure(figsize=(15, 5))
    plt.suptitle(f'Sample Images for Class: {class_name}', fontsize=16)

    for i, image_name in enumerate(selected_images):
        image_path = os.path.join(class_images_path, image_name)
        try:
            img = Image.open(image_path)
            ax = plt.subplot(1, num_samples, i + 1)
            plt.imshow(img, cmap='gray') # Assuming X-ray images are grayscale
            plt.title(image_name.split('.')[0], fontsize=10)
            plt.axis('off')
        except Exception as e:
            print(f"Could not load image {image_name}: {e}")
            continue

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
    plt.show()

print("Function 'display_sample_images' defined.")

In [ ]:
for class_name in expected_classes:
    display_sample_images(class_name, num_samples=4)
    print(f"\nDisplayed sample images for class: {class_name}")

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image
import numpy as np

print("Libraries imported: torch, torchvision.transforms, torch.utils.data.Dataset, torch.utils.data.DataLoader, os, PIL.Image, numpy")

### Preprocessing

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 32 # Define a batch size for later use

# Define image transformations for training
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15), # Data augmentation
    transforms.RandomHorizontalFlip(), # Data augmentation
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1), # Data augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet stats for normalization
])

# Define image transformations for validation and testing (no augmentation)
val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet stats for normalization
])

print("Image transformations defined for training, validation, and testing.")

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, base_dir, class_list, transform=None):
        self.base_dir = base_dir
        self.class_list = class_list
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Create a mapping from class name to numerical label
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(class_list)}

        for class_name in class_list:
            class_images_path = os.path.join(self.base_dir, class_name, 'images')
            if os.path.exists(class_images_path) and os.path.isdir(class_images_path):
                for img_name in os.listdir(class_images_path):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.image_paths.append(os.path.join(class_images_path, img_name))
                        self.labels.append(self.class_to_idx[class_name])
            else:
                print(f"Warning: Image directory for class '{class_name}' not found at {class_images_path}")

        print(f"Initialized CustomImageDataset with {len(self.image_paths)} images across {len(class_list)} classes.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB') # Ensure image is RGB
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

print("CustomImageDataset class defined.")

In [ ]:
full_dataset = CustomImageDataset(base_dir=base_path, class_list=expected_classes, transform=val_test_transforms)
print(f"Full dataset initialized with {len(full_dataset)} images.")

### Train test splitting

In [ ]:
from torch.utils.data import random_split

# Define the split ratios
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# Calculate the number of samples for each split
total_size = len(full_dataset)
train_size = int(train_ratio * total_size)
val_size = int(val_ratio * total_size)
test_size = total_size - train_size - val_size # Ensure all samples are accounted for

# Perform the split
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

print(f"Dataset split into:\n  Training: {len(train_dataset)} images\n  Validation: {len(val_dataset)} images\n  Test: {len(test_dataset)} images")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders created with batch size: {BATCH_SIZE}")
print(f"  Train DataLoader: {len(train_loader)} batches")
print(f"  Validation DataLoader: {len(val_loader)} batches")
print(f"  Test DataLoader: {len(test_loader)} batches")

### EfficientNet-B0

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import numpy as np
import gc

# FREE GPU MEMORY FIRST
torch.cuda.empty_cache()
gc.collect()

# Reduce batch size for memory safety
BATCH_SIZE = 16
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ BATCH_SIZE reduced to {BATCH_SIZE}")

num_classes = len(expected_classes)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load EfficientNet-B0
effnet_model = models.efficientnet_b0(weights='IMAGENET1K_V1')
num_ftrs = effnet_model.classifier[1].in_features
effnet_model.classifier[1] = nn.Linear(num_ftrs, num_classes)
effnet_model = effnet_model.to(device)

# Class weights
class_counts_list = list(refined_class_counts.values())
total_samples = sum(class_counts_list)
class_weights = total_samples / (len(class_counts_list) * np.array(class_counts_list))
class_weights = torch.FloatTensor(class_weights).to(device)

effnet_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
effnet_optimizer = optim.Adam(effnet_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(effnet_optimizer, mode='min', factor=0.5, patience=5)

print("✅ EfficientNet-B0 setup complete")

num_epochs = 30

try:
    for epoch in range(num_epochs):
        print(f"\nEpoch [{epoch+1}/{num_epochs}]")

        # ======================
        # Training phase
        # ======================
        effnet_model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        train_bar = tqdm(train_loader, desc="Training", leave=False)
        for inputs, labels in train_bar:
            inputs = inputs.to(device)
            labels = labels.to(device)

            effnet_optimizer.zero_grad()

            outputs = effnet_model(inputs)
            loss = effnet_criterion(outputs, labels)

            loss.backward()
            effnet_optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

            train_bar.set_postfix(loss=loss.item())

        train_accuracy = 100 * correct_train / total_train
        avg_train_loss = running_loss / len(train_loader)

        # ======================
        # Validation phase
        # ======================
        effnet_model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc="Validation", leave=False)
            for inputs, labels in val_bar:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = effnet_model(inputs)
                loss = effnet_criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

                val_bar.set_postfix(loss=loss.item())

        val_accuracy = 100 * correct_val / total_val
        avg_val_loss = val_loss / len(val_loader)

        # Step scheduler
        scheduler.step(avg_val_loss)

        # ======================
        # Test phase
        # ======================
        test_loss = 0.0
        correct_test = 0
        total_test = 0

        with torch.no_grad():
            test_bar = tqdm(test_loader, desc="Testing", leave=False)
            for inputs, labels in test_bar:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = effnet_model(inputs)
                loss = effnet_criterion(outputs, labels)

                test_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_test += labels.size(0)
                correct_test += (predicted == labels).sum().item()

                test_bar.set_postfix(loss=loss.item())

        test_accuracy = 100 * correct_test / total_test
        avg_test_loss = test_loss / len(test_loader)

        # ======================
        # Epoch summary
        # ======================
        print(
            f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}% | "
            f"Test Loss: {avg_test_loss:.4f}, Test Acc: {test_accuracy:.2f}%"
        )

except KeyboardInterrupt:
    print("\n⛔ Training stopped by user (Ctrl + C). Saving model...")

# ======================
# Save model
# ======================
torch.save(effnet_model.state_dict(), 'effnet_b0_covid_classifier.pth')
print("✅ EfficientNet-B0 model saved to effnet_b0_covid_classifier.pth")


### Result Evaluation

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

# Load trained EfficientNet-B0 model
num_classes = len(expected_classes)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

effnet_model = models.efficientnet_b0(weights=None)
num_ftrs = effnet_model.classifier[1].in_features
effnet_model.classifier[1] = nn.Linear(num_ftrs, num_classes)
effnet_model.load_state_dict(torch.load('effnet_b0_covid_classifier.pth'))
effnet_model = effnet_model.to(device)
effnet_model.eval()

print("✅ EfficientNet-B0 model loaded!")

# Class names
class_names = expected_classes  # ['COVID', 'Normal', 'Viral Pneumonia', 'Lung_Opacity']

# Final evaluation on test set
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    test_bar = tqdm(test_loader, desc="Final Test Evaluation")
    for inputs, labels in test_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = effnet_model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Calculate metrics
test_accuracy = accuracy_score(all_labels, all_preds) * 100
print(f"\n🎯 FINAL TEST ACCURACY: {test_accuracy:.2f}%")

# Classification Report
print("\n📊 CLASSIFICATION REPORT:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# Per-class accuracy
class_accuracies = []
for i, class_name in enumerate(class_names):
    class_mask = all_labels == i
    if np.sum(class_mask) > 0:
        class_acc = accuracy_score(all_labels[class_mask], all_preds[class_mask]) * 100
        class_accuracies.append(class_acc)
        print(f"  {class_name}: {class_acc:.2f}% ({np.sum(class_mask)} samples)")
    else:
        print(f"  {class_name}: No samples")

print(f"\n📈 MACRO AVERAGE ACCURACY: {np.mean(class_accuracies):.2f}%")

# Confusion Matrix
plt.figure(figsize=(12, 10))
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='d')
plt.title(f'Confusion Matrix - EfficientNet-B0\nTest Accuracy: {test_accuracy:.2f}%', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Top-5 error analysis
errors = all_labels != all_preds
if np.sum(errors):
    print(f"\n❌ TOP MISCLASSIFICATIONS ({np.sum(errors)} errors):")
    error_indices = np.where(errors)[0]
    for i in range(min(10, len(error_indices))):
        idx = error_indices[i]
        true_class = class_names[all_labels[idx]]
        pred_class = class_names[all_preds[idx]]
        confidence = np.max(all_probs[idx])
        print(f"  True: {true_class} → Pred: {pred_class} (conf: {confidence:.3f})")

# Probability distribution plot
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.hist(all_probs.max(axis=1), bins=50, alpha=0.7, color='skyblue', edgecolor='black')
plt.title('Prediction Confidence Distribution')
plt.xlabel('Max Probability')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
correct_conf = all_probs[np.arange(len(all_labels)), all_labels].max()
incorrect_conf = all_probs[errors, all_preds[errors]].max()
plt.hist(correct_conf, bins=30, alpha=0.7, label='Correct', color='green')
plt.hist(incorrect_conf, bins=30, alpha=0.7, label='Incorrect', color='red')
plt.title('Confidence: Correct vs Incorrect')
plt.xlabel('Confidence')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 3, 3)
per_class_probs = np.mean(all_probs, axis=0)
plt.bar(class_names, per_class_probs, color='coral', alpha=0.8)
plt.title('Average Predicted Probability per Class')
plt.ylabel('Avg Probability')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n✅ COMPLETE EVALUATION FINISHED!")
print(f"Model saved as: effnet_b0_covid_classifier.pth")
print(f"Peak Test Accuracy: {test_accuracy:.2f}%")


In [ ]:
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = effnet_model(inputs)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_probs.append(probs.cpu())

# Concatenate ONCE
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probs = torch.cat(all_probs).numpy()


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    all_labels,
    all_preds,
    labels=np.arange(len(class_names)),  # force consistent class order
    average=None,
    zero_division=0
)


In [ ]:
x = np.arange(len(class_names))

plt.figure(figsize=(12, 6))
plt.bar(x - 0.25, precision, width=0.25, label='Precision')
plt.bar(x, recall, width=0.25, label='Recall')
plt.bar(x + 0.25, f1, width=0.25, label='F1-score')

plt.xticks(x, class_names, rotation=45)
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.title('Per-Class Precision, Recall, and F1-score (EfficientNet-B0)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
cm_norm = confusion_matrix(all_labels, all_preds, normalize='true')

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix (%) – EfficientNet-B0")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_true_bin = label_binarize(all_labels, classes=np.arange(num_classes))

plt.figure(figsize=(10, 8))

for i, class_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (OvR) – EfficientNet-B0")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
bins = np.linspace(0, 1, 11)
bin_acc, bin_conf = [], []

max_conf = all_probs.max(axis=1)

for i in range(len(bins) - 1):
    mask = (max_conf >= bins[i]) & (max_conf < bins[i+1])
    if mask.any():
        bin_acc.append(np.mean(all_preds[mask] == all_labels[mask]))
        bin_conf.append(np.mean(max_conf[mask]))

plt.figure(figsize=(6, 6))
plt.plot(bin_conf, bin_acc, marker='o', label='EfficientNet-B0')
plt.plot([0, 1], [0, 1], '--', label='Perfect calibration')

plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.title("Reliability Diagram – EfficientNet-B0")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
confidence = all_probs.max(axis=1)

sorted_idx = np.argsort(confidence)
hard_idx = sorted_idx[:5]
easy_idx = sorted_idx[-5:]

def show_samples(indices, title):
    plt.figure(figsize=(15, 3))
    for i, idx in enumerate(indices):
        img, label = test_dataset[idx]
        img = denormalize(img)

        pred = class_names[all_preds[idx]]
        true = class_names[label]
        conf = confidence[idx]

        plt.subplot(1, len(indices), i + 1)
        plt.imshow(img)
        plt.title(f"T:{true}\nP:{pred}\nC:{conf:.2f}")
        plt.axis("off")

    plt.suptitle(title)
    plt.show()

show_samples(easy_idx, "High-Confidence (Easy) Predictions")
show_samples(hard_idx, "Low-Confidence (Hard) Predictions")


In [ ]:
plt.figure(figsize=(12, 6))

for i, class_name in enumerate(class_names):
    class_mask = all_labels == i
    if class_mask.any():
        plt.hist(
            all_probs[class_mask, i],
            bins=30,
            alpha=0.5,
            label=class_name
        )

plt.xlabel("Confidence")
plt.ylabel("Frequency")
plt.title("Class-wise Confidence Distribution – EfficientNet-B0")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
entropy = -np.sum(all_probs * np.log(all_probs + 1e-8), axis=1)

plt.figure(figsize=(8, 5))
plt.hist(entropy, bins=50, color='purple', alpha=0.7)
plt.xlabel("Prediction Entropy")
plt.ylabel("Frequency")
plt.title("Prediction Uncertainty Distribution – EfficientNet-B0")
plt.show()


In [ ]:
error_rates = []

for i, class_name in enumerate(class_names):
    mask = all_labels == i
    if mask.any():
        error = np.mean(all_preds[mask] != all_labels[mask])
        error_rates.append(error)
    else:
        error_rates.append(0)

plt.figure(figsize=(10, 5))
plt.bar(class_names, error_rates, color='crimson', alpha=0.7)
plt.ylabel("Error Rate")
plt.title("Per-Class Error Rate – EfficientNet-B0")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### ConvneXt Tinny

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import numpy as np

# FREE GPU MEMORY
torch.cuda.empty_cache()

# BATCH_SIZE=8 for ConvNeXt (works with 256x256)
BATCH_SIZE = 8
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ ConvNeXt BATCH_SIZE={BATCH_SIZE}")

num_classes = len(expected_classes)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# FIXED: Use ConvNeXt-Tiny (modern CNN, 256x256 OK, ViT-like performance)
convnext_model = timm.create_model('convnext_tiny', pretrained=True, num_classes=num_classes)
convnext_model = convnext_model.to(device)

# Fresh class weights
class_counts_list = list(refined_class_counts.values())
total_samples = sum(class_counts_list)
class_weights_np = total_samples / (len(class_counts_list) * np.array(class_counts_list))
class_weights = torch.FloatTensor(class_weights_np).to(device)

convnext_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
convnext_optimizer = optim.Adam(convnext_model.parameters(), lr=3e-5, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(convnext_optimizer, mode='min', factor=0.5, patience=5)

print("✅ ConvNeXt-Tiny loaded! (256x256 compatible)")

num_epochs = 30

try:
    for epoch in range(num_epochs):
        print(f"\nEpoch [{epoch+1}/{num_epochs}]")

        # ======================
        # Training phase
        # ======================
        convnext_model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        train_bar = tqdm(train_loader, desc="Training", leave=False)
        for inputs, labels in train_bar:
            inputs = inputs.to(device)
            labels = labels.to(device)

            convnext_optimizer.zero_grad()
            outputs = convnext_model(inputs)
            loss = convnext_criterion(outputs, labels)

            loss.backward()
            convnext_optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

            train_bar.set_postfix(loss=loss.item())

        train_accuracy = 100 * correct_train / total_train
        avg_train_loss = running_loss / len(train_loader)

        # ======================
        # Validation phase
        # ======================
        convnext_model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc="Validation", leave=False)
            for inputs, labels in val_bar:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = convnext_model(inputs)
                loss = convnext_criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

                val_bar.set_postfix(loss=loss.item())

        val_accuracy = 100 * correct_val / total_val
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step(avg_val_loss)

        # ======================
        # Test phase
        # ======================
        test_loss = 0.0
        correct_test = 0
        total_test = 0

        with torch.no_grad():
            test_bar = tqdm(test_loader, desc="Testing", leave=False)
            for inputs, labels in test_bar:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = convnext_model(inputs)
                loss = convnext_criterion(outputs, labels)

                test_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_test += labels.size(0)
                correct_test += (predicted == labels).sum().item()

                test_bar.set_postfix(loss=loss.item())

        test_accuracy = 100 * correct_test / total_test
        avg_test_loss = test_loss / len(test_loader)

        print(
            f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}% | "
            f"Test Loss: {avg_test_loss:.4f}, Test Acc: {test_accuracy:.2f}%"
        )

except KeyboardInterrupt:
    print("\n⛔ Training stopped by user. Saving model...")

torch.save(convnext_model.state_dict(), 'convnext_tiny_covid_classifier.pth')
print("✅ ConvNeXt-Tiny saved!")


### Result Evaluation for ConvNeXt-Tiny

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay
from tqdm import tqdm
import timm

# Load trained ConvNeXt-Tiny model
num_classes = len(expected_classes)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

convnext_model = timm.create_model('convnext_tiny', pretrained=False, num_classes=num_classes)
convnext_model.load_state_dict(torch.load('convnext_tiny_covid_classifier.pth'))
convnext_model = convnext_model.to(device)
convnext_model.eval()

print("✅ ConvNeXt-Tiny model loaded!")

# Class names
class_names = expected_classes  # ['COVID', 'Normal', 'Viral Pneumonia', 'Lung_Opacity']

print("🔍 Running final evaluation on test set...")
# Final evaluation on test set
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    test_bar = tqdm(test_loader, desc="Final Test Evaluation")
    for inputs, labels in test_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = convnext_model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Calculate metrics
test_accuracy = accuracy_score(all_labels, all_preds) * 100
print(f"\n🎯 FINAL TEST ACCURACY: {test_accuracy:.2f}%")

# Detailed Classification Report
print("\n📊 CLASSIFICATION REPORT:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# Per-class accuracy
print("\n📈 PER-CLASS ACCURACY:")
class_accuracies = []
for i, class_name in enumerate(class_names):
    class_mask = all_labels == i
    class_samples = np.sum(class_mask)
    if class_samples > 0:
        class_acc = accuracy_score(all_labels[class_mask], all_preds[class_mask]) * 100
        class_accuracies.append(class_acc)
        print(f"  {class_name}: {class_acc:.2f}% ({class_samples:,} samples)")
    else:
        print(f"  {class_name}: No samples")

print(f"\n📊 MACRO AVERAGE ACCURACY: {np.mean(class_accuracies):.2f}%")

# Confusion Matrix
plt.figure(figsize=(12, 10))
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='d', xticks_rotation=45)
plt.title(f'ConvNeXt-Tiny Confusion Matrix\nTest Accuracy: {test_accuracy:.2f}%', 
          fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Error Analysis
errors = all_labels != all_preds
num_errors = np.sum(errors)
print(f"\n❌ ERROR ANALYSIS ({num_errors} errors out of {len(all_labels)}):")
if num_errors > 0:
    error_indices = np.where(errors)[0]
    print("Top 10 Misclassifications:")
    for i in range(min(10, len(error_indices))):
        idx = error_indices[i]
        true_class = class_names[all_labels[idx]]
        pred_class = class_names[all_preds[idx]]
        confidence = np.max(all_probs[idx])
        print(f"  True: {true_class:12s} → Pred: {pred_class:12s} (conf: {confidence:.3f})")

# Visualization Dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Confidence Distribution
axes[0,0].hist(all_probs.max(axis=1), bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0,0].set_title('Prediction Confidence Distribution')
axes[0,0].set_xlabel('Max Probability')
axes[0,0].set_ylabel('Frequency')

# 2. Correct vs Incorrect Confidence
correct_mask = ~errors
incorrect_mask = errors
axes[0,1].hist(all_probs[correct_mask].max(axis=1), bins=30, alpha=0.7, label='Correct', color='green')
axes[0,1].hist(all_probs[incorrect_mask].max(axis=1), bins=30, alpha=0.7, label='Incorrect', color='red')
axes[0,1].set_title('Confidence: Correct vs Incorrect')
axes[0,1].set_xlabel('Confidence')
axes[0,1].legend()

# 3. Per-class prediction probabilities
per_class_probs = np.mean(all_probs, axis=0)
axes[1,0].bar(class_names, per_class_probs, color='coral', alpha=0.8)
axes[1,0].set_title('Average Predicted Probability per Class')
axes[1,0].set_ylabel('Avg Probability')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Per-class accuracy bar chart
axes[1,1].bar(class_names, class_accuracies, color='gold', alpha=0.8, edgecolor='black')
axes[1,1].set_title('Per-Class Accuracy')
axes[1,1].set_ylabel('Accuracy (%)')
axes[1,1].set_ylim(0, 105)
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Summary Table
print("\n" + "="*80)
print("📋 CONVNEXT-TINY FINAL RESULTS SUMMARY")
print("="*80)
print(f"Test Accuracy:        {test_accuracy:.2f}%")
print(f"Macro Avg Accuracy:   {np.mean(class_accuracies):.2f}%")
print(f"Total Test Samples:   {len(all_labels):,}")
print(f"Number of Errors:     {num_errors:,} ({100*num_errors/len(all_labels):.2f}%)")
print(f"Model File:           convnext_tiny_covid_classifier.pth")
print("="*80)

print("\n✅ COMPLETE EVALUATION FINISHED!")
print("Ready for EfficientNet-B0 + ConvNeXt ensemble! 🚀")


In [ ]:
all_preds = []
all_labels = []
all_probs = []

convnext_model.eval()

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = convnext_model(inputs)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_probs.append(probs.cpu())

# Concatenate ONCE
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probs = torch.cat(all_probs).numpy()

assert len(all_preds) == len(all_labels) == len(all_probs)


In [ ]:
from sklearn.metrics import accuracy_score

test_accuracy = accuracy_score(all_labels, all_preds) * 100
print(f"\n🎯 ConvNeXt-Tiny FINAL TEST ACCURACY: {test_accuracy:.2f}%")


In [ ]:
from sklearn.metrics import classification_report

print("\n📊 ConvNeXt-Tiny CLASSIFICATION REPORT:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names,
    digits=4,
    zero_division=0
))


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    all_labels,
    all_preds,
    labels=np.arange(len(class_names)),
    average=None,
    zero_division=0
)

x = np.arange(len(class_names))

plt.figure(figsize=(12, 6))
plt.bar(x - 0.25, precision, width=0.25, label='Precision')
plt.bar(x, recall, width=0.25, label='Recall')
plt.bar(x + 0.25, f1, width=0.25, label='F1-score')

plt.xticks(x, class_names, rotation=45)
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.title("Per-Class Precision, Recall, and F1-score – ConvNeXt-Tiny")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm_norm = confusion_matrix(all_labels, all_preds, normalize='true')

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix (%) – ConvNeXt-Tiny")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_true_bin = label_binarize(all_labels, classes=np.arange(len(class_names)))

plt.figure(figsize=(10, 8))

for i, class_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (One-vs-Rest) – ConvNeXt-Tiny")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
bins = np.linspace(0, 1, 11)
bin_acc, bin_conf = [], []

max_conf = all_probs.max(axis=1)

for i in range(len(bins) - 1):
    mask = (max_conf >= bins[i]) & (max_conf < bins[i+1])
    if mask.any():
        bin_acc.append(np.mean(all_preds[mask] == all_labels[mask]))
        bin_conf.append(np.mean(max_conf[mask]))

plt.figure(figsize=(6, 6))
plt.plot(bin_conf, bin_acc, marker='o', label='ConvNeXt-Tiny')
plt.plot([0, 1], [0, 1], '--', label='Perfect Calibration')

plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.title("Reliability Diagram – ConvNeXt-Tiny")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
confidence = all_probs.max(axis=1)

sorted_idx = np.argsort(confidence)
hard_idx = sorted_idx[:5]
easy_idx = sorted_idx[-5:]

def show_samples(indices, title):
    plt.figure(figsize=(15, 3))
    for i, idx in enumerate(indices):
        img, label = test_dataset[idx]
        img = denormalize(img)

        pred = class_names[all_preds[idx]]
        true = class_names[label]
        conf = confidence[idx]

        plt.subplot(1, len(indices), i + 1)
        plt.imshow(img)
        plt.title(f"T:{true}\nP:{pred}\nC:{conf:.2f}")
        plt.axis("off")

    plt.suptitle(title)
    plt.show()

show_samples(easy_idx, "High-Confidence Predictions – ConvNeXt-Tiny")
show_samples(hard_idx, "Low-Confidence Predictions – ConvNeXt-Tiny")


In [ ]:
plt.figure(figsize=(12, 6))

for i, class_name in enumerate(class_names):
    mask = all_labels == i
    if mask.any():
        plt.hist(
            all_probs[mask, i],
            bins=30,
            alpha=0.5,
            label=class_name
        )

plt.xlabel("Confidence")
plt.ylabel("Frequency")
plt.title("Class-wise Confidence Distribution – ConvNeXt-Tiny")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
entropy = -np.sum(all_probs * np.log(all_probs + 1e-8), axis=1)

plt.figure(figsize=(8, 5))
plt.hist(entropy, bins=50, color='darkorange', alpha=0.7)
plt.xlabel("Prediction Entropy")
plt.ylabel("Frequency")
plt.title("Prediction Uncertainty Distribution – ConvNeXt-Tiny")
plt.show()


In [ ]:
error_rates = []

for i in range(len(class_names)):
    mask = all_labels == i
    if mask.any():
        error_rates.append(np.mean(all_preds[mask] != all_labels[mask]))
    else:
        error_rates.append(0)

plt.figure(figsize=(10, 5))
plt.bar(class_names, error_rates, color='firebrick', alpha=0.7)
plt.ylabel("Error Rate")
plt.title("Per-Class Error Rate – ConvNeXt-Tiny")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Adaptive Weighted ensemble

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay
from tqdm import tqdm
from torchvision import models
import timm

# Load both trained models
num_classes = len(expected_classes)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
class_names = expected_classes

print("🔄 Loading trained models...")
print("Loading EfficientNet-B0...")
effnet_model = models.efficientnet_b0(weights=None)
num_ftrs = effnet_model.classifier[1].in_features
effnet_model.classifier[1] = nn.Linear(num_ftrs, num_classes)
effnet_model.load_state_dict(torch.load('effnet_b0_covid_classifier.pth'))
effnet_model = effnet_model.to(device)
effnet_model.eval()

print("Loading ConvNeXt-Tiny...")
convnext_model = timm.create_model('convnext_tiny', pretrained=False, num_classes=num_classes)
convnext_model.load_state_dict(torch.load('convnext_tiny_covid_classifier.pth'))
convnext_model = convnext_model.to(device)
convnext_model.eval()

print("✅ Both models loaded!")

# Adaptive Weighted Ensemble Evaluation
print("\n🎯 ADAPTIVE WEIGHTED ENSEMBLE EVALUATION")
print("="*70)

all_effnet_probs = []
all_convnext_probs = []
all_labels = []
all_effnet_preds = []
all_convnext_preds = []

# Collect predictions from both models
print("Collecting predictions from both models...")
with torch.no_grad():
    test_bar = tqdm(test_loader, desc="Ensemble Evaluation")
    for inputs, labels in test_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # EfficientNet predictions
        effnet_outputs = effnet_model(inputs)
        effnet_probs = F.softmax(effnet_outputs, dim=1)
        effnet_preds = torch.argmax(effnet_probs, dim=1)
        
        # ConvNeXt predictions
        convnext_outputs = convnext_model(inputs)
        convnext_probs = F.softmax(convnext_outputs, dim=1)
        convnext_preds = torch.argmax(convnext_probs, dim=1)
        
        all_effnet_probs.append(effnet_probs.cpu())
        all_convnext_probs.append(convnext_probs.cpu())
        all_labels.append(labels.cpu())
        all_effnet_preds.append(effnet_preds.cpu())
        all_convnext_preds.append(convnext_preds.cpu())

# Convert to numpy
all_effnet_probs = torch.cat(all_effnet_probs).numpy()
all_convnext_probs = torch.cat(all_convnext_probs).numpy()
all_labels_np = torch.cat(all_labels).numpy()
all_effnet_preds_np = torch.cat(all_effnet_preds).numpy()
all_convnext_preds_np = torch.cat(all_convnext_preds).numpy()

# Calculate individual accuracies
effnet_acc = accuracy_score(all_labels_np, all_effnet_preds_np) * 100
convnext_acc = accuracy_score(all_labels_np, all_convnext_preds_np) * 100

print(f"\n📊 INDIVIDUAL MODEL PERFORMANCE:")
print(f"EfficientNet-B0:  {effnet_acc:.2f}%")
print(f"ConvNeXt-Tiny:    {convnext_acc:.2f}%")

# Adaptive weights based on validation accuracy
total_acc = effnet_acc + convnext_acc
effnet_weight = effnet_acc / total_acc
convnext_weight = convnext_acc / total_acc

print(f"\n⚖️  ADAPTIVE WEIGHTS:")
print(f"EfficientNet-B0:  {effnet_weight:.3f} ({effnet_acc:.2f}%)")
print(f"ConvNeXt-Tiny:    {convnext_weight:.3f} ({convnext_acc:.2f}%)")

# Ensemble predictions (adaptive weighted average)
ensemble_probs = effnet_weight * all_effnet_probs + convnext_weight * all_convnext_probs
ensemble_preds = np.argmax(ensemble_probs, axis=1)
ensemble_accuracy = accuracy_score(all_labels_np, ensemble_preds) * 100

print(f"\n🎉 ENSEMBLE PERFORMANCE:")
print(f"Adaptive Weighted Ensemble: {ensemble_accuracy:.2f}%")
print(f"Improvement: +{ensemble_accuracy-max(effnet_acc, convnext_acc):+.2f}%")

# Detailed metrics
print(f"\n📋 DETAILED CLASSIFICATION REPORT:")
print(classification_report(all_labels_np, ensemble_preds, target_names=class_names, digits=4))

# Confusion Matrix for Ensemble
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
cm_effnet = confusion_matrix(all_labels_np, all_effnet_preds_np)
sns.heatmap(cm_effnet, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'EfficientNet-B0\n{effnet_acc:.2f}%')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.subplot(1, 3, 2)
cm_convnext = confusion_matrix(all_labels_np, all_convnext_preds_np)
sns.heatmap(cm_convnext, annot=True, fmt='d', cmap='Greens', 
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'ConvNeXt-Tiny\n{convnext_acc:.2f}%')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.subplot(1, 3, 3)
cm_ensemble = confusion_matrix(all_labels_np, ensemble_preds)
sns.heatmap(cm_ensemble, annot=True, fmt='d', cmap='Reds', 
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Ensemble (Adaptive Weighted)\n{ensemble_accuracy:.2f}%')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

# Per-class improvement analysis
print(f"\n📈 PER-CLASS IMPROVEMENT:")
class_accuracies = {}
for i, class_name in enumerate(class_names):
    class_mask = all_labels_np == i
    if np.sum(class_mask) > 0:
        effnet_class_acc = accuracy_score(all_labels_np[class_mask], all_effnet_preds_np[class_mask]) * 100
        convnext_class_acc = accuracy_score(all_labels_np[class_mask], all_convnext_preds_np[class_mask]) * 100
        ensemble_class_acc = accuracy_score(all_labels_np[class_mask], ensemble_preds[class_mask]) * 100
        
        class_accuracies[class_name] = {
            'effnet': effnet_class_acc,
            'convnext': convnext_class_acc,
            'ensemble': ensemble_class_acc
        }
        
        best_single = max(effnet_class_acc, convnext_class_acc)
        improvement = ensemble_class_acc - best_single
        
        print(f"{class_name:14s}: {effnet_class_acc:6.1f}% | {convnext_class_acc:6.1f}% | {ensemble_class_acc:6.1f}% | +{improvement:+.1f}%")

# Performance comparison bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Overall accuracy comparison
models = ['EfficientNet-B0', 'ConvNeXt-Tiny', 'Ensemble']
accuracies = [effnet_acc, convnext_acc, ensemble_accuracy]
colors = ['skyblue', 'lightgreen', 'gold']
bars1 = ax1.bar(models, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
ax1.set_title('Overall Test Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_ylabel('Accuracy (%)')
ax1.set_ylim(0, 102)

# Add value labels on bars
for bar, acc in zip(bars1, accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold')

# Per-class comparison (macro avg)
class_accs = list(class_accuracies.values())
macro_effnet = np.mean([acc['effnet'] for acc in class_accs])
macro_convnext = np.mean([acc['convnext'] for acc in class_accs])
macro_ensemble = np.mean([acc['ensemble'] for acc in class_accs])

models_macro = ['EffNet Macro', 'ConvNeXt Macro', 'Ensemble Macro']
macro_accs = [macro_effnet, macro_convnext, macro_ensemble]
bars2 = ax2.bar(models_macro, macro_accs, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
ax2.set_title('Macro-Average Per-Class Accuracy', fontsize=14, fontweight='bold')
ax2.set_ylabel('Macro Accuracy (%)')
ax2.set_ylim(0, 102)

for bar, acc in zip(bars2, macro_accs):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# FINAL SUMMARY
print(f"\n" + "="*80)
print("🏆 ADAPTIVE WEIGHTED ENSEMBLE FINAL RESULTS")
print("="*80)
print(f"Individual Models:")
print(f"  EfficientNet-B0:  {effnet_acc:6.2f}% (weight: {effnet_weight:.3f})")
print(f"  ConvNeXt-Tiny:    {convnext_acc:6.2f}% (weight: {convnext_weight:.3f})")
print(f"\n🎯 ENSEMBLE RESULT: {ensemble_accuracy:.2f}%")
print(f"   ↑ Improvement:   +{(ensemble_accuracy-max(effnet_acc, convnext_acc)):+.2f}% points")
print(f"   ↑ Total Gain:    +{(ensemble_accuracy - min(effnet_acc, convnext_acc)):.2f}% points")
print("="*80)


In [ ]:
assert (
    len(all_labels_np)
    == len(all_effnet_preds_np)
    == len(all_convnext_preds_np)
    == len(ensemble_preds)
), "❌ Length mismatch in ensemble evaluation arrays!"


In [ ]:
def per_class_accuracy(labels, preds, num_classes):
    accs = []
    for i in range(num_classes):
        mask = labels == i
        accs.append(np.mean(preds[mask] == labels[mask]) * 100 if mask.any() else 0)
    return accs

effnet_class_acc = per_class_accuracy(all_labels_np, all_effnet_preds_np, num_classes)
convnext_class_acc = per_class_accuracy(all_labels_np, all_convnext_preds_np, num_classes)
ensemble_class_acc = per_class_accuracy(all_labels_np, ensemble_preds, num_classes)

x = np.arange(num_classes)
width = 0.25

plt.figure(figsize=(12, 6))
plt.bar(x - width, effnet_class_acc, width, label="EfficientNet-B0")
plt.bar(x, convnext_class_acc, width, label="ConvNeXt-Tiny")
plt.bar(x + width, ensemble_class_acc, width, label="Ensemble")

plt.xticks(x, class_names, rotation=45)
plt.ylabel("Accuracy (%)")
plt.title("Per-Class Accuracy Comparison")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

plt.figure(figsize=(18, 5))

for i, (name, preds) in enumerate([
    ("EfficientNet-B0", all_effnet_preds_np),
    ("ConvNeXt-Tiny", all_convnext_preds_np),
    ("Adaptive Ensemble", ensemble_preds)
]):
    cm = confusion_matrix(all_labels_np, preds, normalize="true")

    plt.subplot(1, 3, i + 1)
    sns.heatmap(
        cm, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(name)
    plt.xlabel("Predicted")
    plt.ylabel("True")

plt.tight_layout()
plt.show()


In [ ]:
fixed_by_ensemble = (
    (all_effnet_preds_np != all_labels_np) &
    (all_convnext_preds_np != all_labels_np) &
    (ensemble_preds == all_labels_np)
)

print(f"✅ Ensemble corrected {fixed_by_ensemble.sum()} samples")

idxs = np.where(fixed_by_ensemble)[0][:5]

plt.figure(figsize=(15, 3))
for i, idx in enumerate(idxs):
    img, label = test_dataset[idx]
    img = denormalize(img)

    plt.subplot(1, len(idxs), i + 1)
    plt.imshow(img)
    plt.title(f"True: {class_names[label]}\nEnsemble ✓")
    plt.axis("off")

plt.suptitle("Samples Corrected by Adaptive Ensemble")
plt.show()


In [ ]:
effnet_error_rates = []
convnext_error_rates = []
ensemble_error_rates = []

for i in range(num_classes):
    mask = all_labels_np == i

    if mask.any():
        eff_err = np.mean(all_effnet_preds_np[mask] != all_labels_np[mask])
        conv_err = np.mean(all_convnext_preds_np[mask] != all_labels_np[mask])
        ens_err = np.mean(ensemble_preds[mask] != all_labels_np[mask])
    else:
        eff_err = conv_err = ens_err = 0.0

    effnet_error_rates.append(eff_err * 100)
    convnext_error_rates.append(conv_err * 100)
    ensemble_error_rates.append(ens_err * 100)


In [ ]:
x = np.arange(num_classes)
width = 0.25

plt.figure(figsize=(13, 6))

plt.bar(x - width, effnet_error_rates, width, label="EfficientNet-B0", color="#4C72B0")
plt.bar(x, convnext_error_rates, width, label="ConvNeXt-Tiny", color="#55A868")
plt.bar(x + width, ensemble_error_rates, width, label="Adaptive Ensemble", color="#C44E52")

plt.xticks(x, class_names, rotation=45)
plt.ylabel("Error Rate (%)")
plt.title("Per-Class Error Rate Comparison")
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
error_reduction = []

for i in range(num_classes):
    best_single = min(effnet_error_rates[i], convnext_error_rates[i])
    reduction = best_single - ensemble_error_rates[i]
    error_reduction.append(reduction)


In [ ]:
plt.figure(figsize=(12, 5))

bars = plt.bar(class_names, error_reduction, color="gold", edgecolor="black")

plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Error Reduction (%)")
plt.title("Error Reduction Achieved by Adaptive Ensemble")
plt.xticks(rotation=45)

for bar, val in zip(bars, error_reduction):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        val + (0.5 if val >= 0 else -1),
        f"{val:+.2f}",
        ha="center",
        va="bottom" if val >= 0 else "top",
        fontweight="bold"
    )

plt.tight_layout()
plt.show()
print("\n📋 PER-CLASS ERROR RATE SUMMARY (%)")
print("=" * 70)
print(f"{'Class':15s} | {'EffNet':>8s} | {'ConvNeXt':>9s} | {'Ensemble':>9s}")
print("-" * 70)

for i, cls in enumerate(class_names):
    print(
        f"{cls:15s} | "
        f"{effnet_error_rates[i]:8.2f} | "
        f"{convnext_error_rates[i]:9.2f} | "
        f"{ensemble_error_rates[i]:9.2f}"
    )

print("=" * 70)


### Explainable AI - Grad-Cam

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, inp, out):
            self.activations = out.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate(self, x, class_idx=None):
        self.model.zero_grad()
        logits = self.model(x)

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        score = logits[:, class_idx]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)
        cam = F.relu(cam)

        cam = cam.squeeze().cpu().numpy()
        cam = cv2.resize(cam, (x.size(3), x.size(2)))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam, class_idx
print("Grad-Cam class implemented")


In [ ]:
eff_target_layer = effnet_model.features[-1]
print(".")

In [ ]:
convnext_target_layer = convnext_model.stages[-1].blocks[-1].conv_dw
print("done")

In [ ]:
image_tensor, label = test_dataset[0]
input_tensor = image_tensor.unsqueeze(0).to(device)


In [ ]:
eff_cam_gen = GradCAM(effnet_model, eff_target_layer)
eff_cam, eff_pred = eff_cam_gen.generate(input_tensor)


In [ ]:
conv_cam_gen = GradCAM(convnext_model, convnext_target_layer)
conv_cam, conv_pred = conv_cam_gen.generate(input_tensor)


In [ ]:
def denormalize(img_tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = std * img + mean
    return np.clip(img, 0, 1)

image = denormalize(image_tensor)


In [ ]:
def overlay_cam(image, cam):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    overlay = 0.5 * image + 0.5 * heatmap
    return np.clip(overlay, 0, 1)

eff_overlay = overlay_cam(image, eff_cam)
conv_overlay = overlay_cam(image, conv_cam)


In [ ]:
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
plt.title("Original X-ray")
plt.imshow(image)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("EfficientNet-B0 Grad-CAM")
plt.imshow(eff_overlay)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("ConvNeXt-Tiny Grad-CAM")
plt.imshow(conv_overlay)
plt.axis("off")

plt.show()


In [ ]:
ensemble_cam = 0.5 * eff_cam + 0.5 * conv_cam
ensemble_overlay = overlay_cam(image, ensemble_cam)

plt.figure(figsize=(6, 5))
plt.title("Ensemble Grad-CAM (Averaged)")
plt.imshow(ensemble_overlay)
plt.axis("off")
plt.show()


In [ ]:
import random

def visualize_multiple_samples(test_dataset, n_samples=6):
    indices = random.sample(range(len(test_dataset)), n_samples)

    plt.figure(figsize=(18, 3 * n_samples))

    for i, idx in enumerate(indices):
        image_tensor, label = test_dataset[idx]
        input_tensor = image_tensor.unsqueeze(0).to(device)

        # Grad-CAM
        eff_cam, _ = eff_cam_gen.generate(input_tensor)
        conv_cam, _ = conv_cam_gen.generate(input_tensor)

        image = denormalize(image_tensor)
        eff_overlay = overlay_cam(image, eff_cam)
        conv_overlay = overlay_cam(image, conv_cam)

        plt.subplot(n_samples, 3, 3*i + 1)
        plt.imshow(image)
        plt.title(f"Original\nGT: {expected_classes[label]}")
        plt.axis("off")

        plt.subplot(n_samples, 3, 3*i + 2)
        plt.imshow(eff_overlay)
        plt.title("EfficientNet-B0")
        plt.axis("off")

        plt.subplot(n_samples, 3, 3*i + 3)
        plt.imshow(conv_overlay)
        plt.title("ConvNeXt-Tiny")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_multiple_samples(test_dataset, n_samples=5)


In [ ]:
def visualize_correct_vs_wrong(test_dataset, max_samples=3):
    correct, wrong = [], []

    with torch.no_grad():
        for i in range(len(test_dataset)):
            image, label = test_dataset[i]
            input_tensor = image.unsqueeze(0).to(device)

            eff_logits = effnet_model(input_tensor)
            conv_logits = convnext_model(input_tensor)

            # Simple ensemble (average)
            logits = (eff_logits + conv_logits) / 2
            pred = logits.argmax(dim=1).item()

            if pred == label and len(correct) < max_samples:
                correct.append(i)
            elif pred != label and len(wrong) < max_samples:
                wrong.append(i)

            if len(correct) == max_samples and len(wrong) == max_samples:
                break

    def plot_group(indices, title):
        plt.figure(figsize=(15, 4))
        for i, idx in enumerate(indices):
            img, lbl = test_dataset[idx]
            input_tensor = img.unsqueeze(0).to(device)

            eff_cam, _ = eff_cam_gen.generate(input_tensor)
            image = denormalize(img)
            overlay = overlay_cam(image, eff_cam)

            plt.subplot(1, max_samples, i+1)
            plt.imshow(overlay)
            plt.title(expected_classes[lbl])
            plt.axis("off")
        plt.suptitle(title)
        plt.show()

    plot_group(correct, "Correct Predictions (EfficientNet Grad-CAM)")
    plot_group(wrong, "Misclassified Samples (EfficientNet Grad-CAM)")


In [ ]:
visualize_correct_vs_wrong(test_dataset)


In [ ]:
def visualize_ensemble_cam(idx):
    image_tensor, label = test_dataset[idx]
    input_tensor = image_tensor.unsqueeze(0).to(device)

    eff_cam, _ = eff_cam_gen.generate(input_tensor)
    conv_cam, _ = conv_cam_gen.generate(input_tensor)

    ensemble_cam = 0.5 * eff_cam + 0.5 * conv_cam

    image = denormalize(image_tensor)

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(overlay_cam(image, eff_cam))
    plt.title("EfficientNet-B0")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(overlay_cam(image, conv_cam))
    plt.title("ConvNeXt-Tiny")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(overlay_cam(image, ensemble_cam))
    plt.title("Ensemble CAM")
    plt.axis("off")

    plt.show()


In [ ]:
visualize_ensemble_cam(idx=10)
